In [ ]:
def r_factor(vol_ideal=None, vol_real=None, convex_hull=None, sphere_thickness=1):
    """
    R-factor of two diffraction patterns. The sum within each resolution sphere is calculated including the maximum resolution of the sphere.
    """
    F_ideal = np.sqrt(vol_ideal)
    F_real = np.sqrt(vol_real)

    im_dim = np.array(F_ideal.shape)
    im_center = np.array(np.unravel_index(np.argmax(F_ideal), F_ideal.shape))
    num_spheres = int(min((im_dim - im_center) / sphere_thickness))

    r = np.arange(
        sphere_thickness, (num_spheres + 1) * sphere_thickness, sphere_thickness
    )
    x, y, z = np.meshgrid(
        np.arange(F_ideal.shape[0]),
        np.arange(F_ideal.shape[1]),
        np.arange(F_ideal.shape[2]),
        indexing="ij",
    )
    spherical_mask = (x - im_center[0]) ** 2 + (y - im_center[1]) ** 2 + (
        z - im_center[2]
    ) ** 2 <= r[:, None, None, None] ** 2

    qs_arr = []
    rf_nb_arr_2 = []
    rf_nb_arr = []
    n_points_convex_hull = convex_hull.sum()
    fft_shift_convex_hull = fft.fftshift(convex_hull)
    rf_3d = np.zeros(shape=F_real.shape)
    print(n_points_convex_hull)
    
    for q_vals in range(num_spheres):
        qs = spherical_mask[q_vals] * (~np.isnan(F_real))

        qs_arr.append(qs.sum())
        
        F_ideal_q = F_ideal[qs]
        sum_F_ideal_q = np.sum(F_ideal_q)

        F_real_q = F_real[qs]
        sum_F_real_q = np.sum(F_real_q)

        #abs_diff = np.abs(F_real / sum_F_real_q - F_ideal / sum_F_ideal_q)
        rf_3d[qs] = np.abs(F_real[qs] / sum_F_real_q - F_ideal[qs] / sum_F_ideal_q)
        rf_nb_arr.append(np.sum(rf_3d[qs]))

        #rf_nb_arr.append(np.sum(np.abs(F_real[qs] / sum_F_real_q - F_ideal[qs] / sum_F_ideal_q)))
        #abs_diff_smooth = np.abs(fft.fftn(abs_diff_ft * fft_shift_convex_hull))
        #abs_diff_smooth /= n_points_convex_hull
            
        #r_val_q = np.sum(abs_diff_smooth[qs])
        #rf_nb_arr.append(np.sum(abs_diff[qs]))
        #rf_arr.append(r_val_q)

    #rf_3d[np.isnan(rf_3d)] = 0.0
    #rf_3d_ft = fft.ifftn(rf_3d)
    #rf_3d_smooth = np.abs(fft.fftn(rf_3d_ft * fft_shift_convex_hull))

    print((np.abs(rf_3d)**2).sum())
    #print((np.abs(rf_3d_smooth)**2).sum())

    for q_vals in range(num_spheres):
        qs = spherical_mask[q_vals] * (~np.isnan(F_real))
        rf_nb_arr_2.append(np.sum(rf_3d[qs]))
        #rf_arr.append(np.sum(rf_3d_smooth[qs]))
        #print(f'Non-blurred area: {np.trapz(abs_diff).sum()} - Blurred area: {np.trapz(abs_diff_smooth).sum()} \n Difference area: {np.abs(np.trapz(abs_diff).sum()-np.trapz(abs_diff_smooth).sum())}')

    #return np.array(rf_nb_arr)#, np.array(rf_nb_arr)#, abs_diff, abs_diff_smooth
    return np.array(rf_nb_arr), np.array(rf_nb_arr_2), np.array(qs_arr) #, abs_diff, abs_diff_smooth

In [ ]:
mrc_name = files_base_dir

# Opening undamaged "ideal" Fourier intensities
with mrcfile.open('r_factors_4x/gt/gt_4x_2.mrc', mode='r') as f_ideal:
    intens_ideal = f_ideal.data
intens_ideal = np.array(intens_ideal)

e_photon_eV = 9000
lambda_photon = (h * c) / (e_photon_eV * e)
d_detector = 0.5
s_pixel = 800e-6 # 800e-6 for 4x and 1200e-6 for 6x

dim = intens_ideal.shape[0]
pixel_num = dim - dim//2
theta_pixel = 0.5 * np.arctan((pixel_num*s_pixel)/d_detector)
resolution = lambda_photon/(2.0*np.sin(theta_pixel))
pix_real = 0.5 * resolution
voxel_size = pix_real

pix_emc = 1 / (intens_ideal.shape[0] * voxel_size * 1e9) # in nm^-1
rad_sh = 5

if rad_sh == 1:
    write_text(f'Size of resolution sphere: {rad_sh} voxel(s) or {pix_emc} nm^-1\n')
else:
    pix_emc *= rad_sh
    write_text(f'Size of resolution sphere: {rad_sh} voxel(s) or {pix_emc} nm^-1\n')
    
# Looping over all aligned Fourier intensities
names_r = []
r_factors = []
emc_list = []

for f in range(len(files_base_dir)):
    # Opening damaged "real" Fourier intensities
    with mrcfile.open(files_base_dir[f], mode='r') as f_real:
        intens_real = f_real.data
    intens_real = np.array(intens_real)
    emc_list.append(intens_real)

    # Opening convex hull of reconstructions
    convex_hull = np.load(convex_hull_dir[f])
    
    rfs = r_factor(vol_ideal=intens_ideal, vol_real=intens_real, sphere_thickness=rad_sh)
    r_factors.append(rfs)
    
    f_names = files_base_dir[f].split(sep='/')[1].split(sep='.mrc')[0][:]
    names_r.append(f_names)

r_factors = np.array(r_factors)
names_r = np.array(names_r)
emc_list = np.array(emc_list)

In [ ]:
rfs, rfs_2, qs_arr = r_factor(vol_ideal=intens_ideal, vol_real=intens_real, convex_hull=convex_hull, sphere_thickness=5)

In [ ]:
rfs.shape

In [ ]:
plt.plot(rfs,'x-')
plt.plot(rfs_2,'o-');
#plt.plot(rfs_nb);

In [ ]:
plt.plot(qs_arr[:10], 'x-');

In [ ]:
plt.figure(1)
plt.imshow(rf_3d[:,:,374//2]**0.2, cmap='viridis')
plt.xticks([])
plt.yticks([])
plt.colorbar()

plt.figure(2)
plt.imshow(rf_3d_smooth[:,:,374//2]**0.2, cmap='viridis')
plt.xticks([])
plt.yticks([])
plt.colorbar();